In [ ]:
!pip install -q ultralytics

In [ ]:
# Connect Google Drive and set up folders
import os
import glob
from google.colab import drive

drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/ECS-174-Project'
VIDEO_DIR = os.path.join(BASE, 'original-footage')
ANNOT_DIR = os.path.join(BASE, 'annotated-footage')
CSV_DIR = os.path.join(BASE, 'annotated-csv')
PATH_DIR = os.path.join(BASE, 'path-image')

for d in (ANNOT_DIR, CSV_DIR, PATH_DIR):
    os.makedirs(d, exist_ok=True)

VIDEO_EXTS = ('.mp4', '.mov', '.m4v', '.avi')
VIDEOS = sorted(f for f in glob.glob(os.path.join(VIDEO_DIR, '*'))
                if f.lower().endswith(VIDEO_EXTS))
print(f'Found {len(VIDEOS)} videos in {VIDEO_DIR}')

In [ ]:
# Model, settings, and helper functions
import time
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from ultralytics import YOLO

FRAME_STRIDE = 3   # process every 3rd frame (~10 fps from 30 fps video)
CONF_OK = 0.5      # keypoint confidence counted as 'tracked'

# COCO keypoint indices
L_ELBOW, R_ELBOW, L_WRIST, R_WRIST = 7, 8, 9, 10
COLUMNS = ['frame', 'person_found', 'lhand_x', 'lhand_y', 'lwrist_conf',
           'rhand_x', 'rhand_y', 'rwrist_conf']

model = YOLO('yolov8m-pose.pt')


def hand_point(wrist, elbow):
    """Estimate the hand position by extending elbow->wrist by ~1/3 of the forearm."""
    return wrist + 0.33 * (wrist - elbow)


def video_info(video):
    cap = cv2.VideoCapture(video)
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    cap.release()
    return n, fps, w, h


def video_stats(df):
    return {
        'frames': len(df),
        'climber_detected': df.person_found.mean(),
        'left_wrist_ok': (df.lwrist_conf >= CONF_OK).mean(),
        'right_wrist_ok': (df.rwrist_conf >= CONF_OK).mean(),
    }


def save_path_image(video, df, name):
    """Draw the estimated hand paths over the first frame of the video."""
    cap = cv2.VideoCapture(video)
    ok, first = cap.read()
    cap.release()
    fig, ax = plt.subplots(figsize=(8, 10))
    if ok:
        ax.imshow(cv2.cvtColor(first, cv2.COLOR_BGR2RGB))
    good_l = df.lwrist_conf >= CONF_OK
    good_r = df.rwrist_conf >= CONF_OK
    ax.plot(df.lhand_x[good_l], df.lhand_y[good_l], '.-', color='cyan', ms=3, lw=0.8, label='left hand')
    ax.plot(df.rhand_x[good_r], df.rhand_y[good_r], '.-', color='magenta', ms=3, lw=0.8, label='right hand')
    ax.legend()
    ax.set_title(f'Estimated hand paths: {name}')
    ax.axis('off')
    fig.savefig(os.path.join(PATH_DIR, name + '_hand_paths.png'), dpi=120, bbox_inches='tight')
    plt.close(fig)


def process_video(video, name, n_frames):
    rows = []
    results = model.predict(video, stream=True, vid_stride=FRAME_STRIDE, save=True,
                            project=BASE, name='annotated-footage', exist_ok=True,
                            verbose=False)
    bar = tqdm(results, total=n_frames // FRAME_STRIDE + 1, desc=name, unit='frame')
    for i, r in enumerate(bar):
        frame = i * FRAME_STRIDE
        if r.keypoints is None or len(r.boxes) == 0:
            rows.append([frame, 0, np.nan, np.nan, 0.0, np.nan, np.nan, 0.0])
            continue
        # If several people are detected, keep the largest box
        areas = r.boxes.xywh[:, 2] * r.boxes.xywh[:, 3]
        k = int(areas.argmax())
        xy = r.keypoints.xy[k].cpu().numpy()
        conf = r.keypoints.conf[k].cpu().numpy()
        lh = hand_point(xy[L_WRIST], xy[L_ELBOW])
        rh = hand_point(xy[R_WRIST], xy[R_ELBOW])
        rows.append([frame, 1, lh[0], lh[1], conf[L_WRIST], rh[0], rh[1], conf[R_WRIST]])
    bar.close()
    df = pd.DataFrame(rows, columns=COLUMNS)
    save_path_image(video, df, name)
    df.to_csv(os.path.join(CSV_DIR, name + '_hands.csv'), index=False)
    return video_stats(df)

In [ ]:
# Process every video (skip ones that are already done)
total = len(VIDEOS)
run_start = time.time()
for n, video in enumerate(VIDEOS, 1):
    name = os.path.splitext(os.path.basename(video))[0]
    if os.path.exists(os.path.join(CSV_DIR, name + '_hands.csv')):
        print(f'[{n}/{total}] {name}: already done, skipping')
        continue
    n_frames, fps, w, h = video_info(video)
    print(f'[{n}/{total}] {name}: {w}x{h}, {fps:.0f} fps, {n_frames / fps:.0f}s long')
    start = time.time()
    try:
        s = process_video(video, name, n_frames)
        print(f"    done: climber {s['climber_detected']:.0%} | left wrist {s['left_wrist_ok']:.0%} | "
              f"right wrist {s['right_wrist_ok']:.0%} ({time.time() - start:.0f}s)")
    except Exception as e:
        print(f'    FAILED ({e})')
print(f'\nFinished in {(time.time() - run_start) / 60:.0f} min')

In [ ]:
# Summary of all processed videos
csvs = sorted(glob.glob(os.path.join(CSV_DIR, '*_hands.csv')))
pct = {c: '{:.0%}' for c in ['climber_detected', 'left_wrist_ok', 'right_wrist_ok']}
if not csvs:
    weak = None
    print(f'No results found in {CSV_DIR}. Check the output of the cell above.')
else:
    summary = pd.DataFrame([{'video': os.path.basename(f).replace('_hands.csv', ''),
                             **video_stats(pd.read_csv(f))} for f in csvs])
    summary.to_csv(os.path.join(CSV_DIR, 'summary.csv'), index=False)
    weak = summary[(summary.left_wrist_ok < 0.6) | (summary.right_wrist_ok < 0.6)]
    print(f'{len(summary)} videos processed; {len(weak)} have a wrist tracked in under 60% of frames')
    display(summary.style.format(pct))

In [ ]:
# Videos to look at more closely (a wrist tracked in under 60% of frames)
if weak is not None:
    display(weak.style.format(pct))